# NYC Weather Bronze Ingestion Pipeline

### Purpose
This pipeline ingests **hourly weather data** for NYC boroughs from the Open-Meteo Archive API, validates schema and data quality, applies deduplication logic, and writes results into Delta tables. It is designed as a **batch-only ingestion** process with auditability, provenance tracking, and quarantine handling for invalid records.

---

### Phase 1: Setup
- Initializes Spark session and runtime identifiers (`ingest_run_id`, `today`, `now`).
- Defines Lakehouse paths for Bronze, Silver, and Gold layers.
- Creates directories using Fabric’s file system API to ensure consistent storage structure.
- Configures borough coordinates for Manhattan, Brooklyn, Queens, Bronx, and Staten Island.

---

### Phase 2: Smart Fetch with Retry
- Implements `fetch_with_retry` function:
  - Retries API calls up to 3 times with backoff.
  - Handles transient network/API errors gracefully.
- Determines fetch range:
  - If `bronze_weather_nyc` exists → start from the day after the latest ingested timestamp.
  - Else → start from historical baseline (`2026-01-01`).
- Fetches only **missing days + current day**, avoiding redundant downloads.
- Saves results as **raw JSON snapshots** per borough for audit trail.

---

### Phase 3: Validation & Transformation
- **Validation (`ingest_weather_to_bronze`)**:
  - Reads raw JSON into Spark.
  - Explodes hourly arrays into tabular rows.
  - Casts fields into correct numeric types (`double`, `int`).
  - Flags invalid records:
    - Timezone mismatch (must be `America/New_York`).
    - Missing critical fields (`timestamp`, `temperature_2m`, `precipitation`, `wind_speed_10m`, `weather_code`).
  - Splits into **good records** vs. **quarantine records**.
  - Adds provenance metadata (`borough`, `source`, `ingest_run_id`, `ingested_at`, `ingest_date`).

---

### Phase 3b: Write to Delta Tables
- **Good Records**:
  - Union across boroughs.
  - Deduplicate within batch on `(borough, timestamp)`.
  - Append to `bronze_weather_nyc` partitioned by `ingest_date`.
  - Run **safety net deduplication** across full table:
    - Window function keeps latest record per `(borough, timestamp)` by `ingested_at`.
    - Rewrites table if duplicates are found.

- **Quarantine Records**:
  - Union across boroughs.
  - Append to `bronze_quarantine_weather_nyc` partitioned by `ingest_date`.
  - Ensures invalid records are preserved for auditability.

---

### Phase 4: QA Queries
The pipeline includes SQL queries to validate ingestion quality and provide audit trails:

1. **Record integrity**: Total rows, distinct boroughs, ingestion runs.  
2. **Borough coverage**: Record counts, earliest and latest timestamps per borough.  
3. **Null checks**: Verifies critical fields (`timestamp`, `temperature_2m`, `precipitation`, `wind_speed_10m`, `borough`) are populated. 
4. **Duplicate check**: Ensures no duplicate `(borough, timestamp)` rows remain.  
5. **Expected row count** : Compares actual vs. expected rows:  Expected = `(days since Jan 1, 2026 × 24 hours × 5 boroughs)`  Flags anomalies if actual count deviates by >5%.  


### Phase 1: Setup

In [2]:
import datetime, uuid, requests, json, time
from functools import reduce
from pyspark.sql.types import DoubleType, IntegerType, StringType
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from datetime import date
from notebookutils import mssparkutils
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

today = datetime.date.today()
now = datetime.datetime.utcnow()
ingest_run_id = "run_" + str(uuid.uuid4())

print(f"Date: {today}  |  Run ID: {ingest_run_id}")

StatementMeta(, 35d9bb74-2302-4cc6-8a16-6eee6d3f81e1, 4, Finished, Available, Finished, False)

Date: 2026-04-21  |  Run ID: run_25c8f7d1-d653-4bb2-9672-53cee3d6739f


In [3]:
LAKEHOUSE = "test_lakehouse_1"
BASE_PATH = "Files/project_data"

PATHS = {
    "bronze_weather": f"{BASE_PATH}/bronze/weather",
    "bronze_311":     f"{BASE_PATH}/bronze/311",
    "bronze_traffic": f"{BASE_PATH}/bronze/traffic",
    "silver":         f"{BASE_PATH}/silver",
    "gold":           f"{BASE_PATH}/gold",
}

for name, path in PATHS.items():
    try:
        mssparkutils.fs.mkdirs(path)
    except Exception as e:
        print(f"{name} creation issue: {e}")

print(f"All folders ready under {BASE_PATH}")

StatementMeta(, 35d9bb74-2302-4cc6-8a16-6eee6d3f81e1, 5, Finished, Available, Finished, False)

All folders ready under Files/project_data


### Phase 2: Data Fetching

In [4]:
def fetch_with_retry(url, params, max_retries=3, backoff=5):
    """
    Perform an HTTP GET request with retry logic.

    Args
        url : str
            The API endpoint to call.
        params : dict
            Query parameters for the request.
        max_retries : int, optional
            Maximum number of retry attempts (default = 3).
        backoff : int, optional
            Delay in seconds between retries (default = 5).

    Returns
        dict
            JSON response from the API if successful.

    Raises
        Exception if all retry attempts fail.
    """

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, params=params, timeout=120)
            response.raise_for_status()
            print(f"Success on attempt {attempt}")
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                time.sleep(backoff)
    raise Exception("All retries failed")

boroughs = {
    "Manhattan":    {"latitude": 40.7128, "longitude": -74.0060},
    "Brooklyn":     {"latitude": 40.6782, "longitude": -73.9442},
    "Queens":       {"latitude": 40.7282, "longitude": -73.7949},
    "Bronx":        {"latitude": 40.8448, "longitude": -73.8648},
    "StatenIsland": {"latitude": 40.5795, "longitude": -74.1502},
}

HISTORICAL_START = datetime.date(2026, 1, 1)

# Determine what date range we need to fetch 
# Check if the Bronze table already exists and find the latest date in it
try:
    df_existing = spark.table("bronze_weather_nyc")
    latest_row = df_existing.agg(F.max("timestamp").alias("latest")).collect()[0]
    latest_ts = latest_row["latest"]

    if latest_ts is not None:
        # Start from the day AFTER the latest data we already have
        fetch_start = (latest_ts.date() + datetime.timedelta(days=1))
        existing_count = df_existing.count()
        print(f"Existing bronze_weather_nyc: {existing_count:,} rows")
        print(f"Latest data: {latest_ts}")
        print(f"Will fetch from: {fetch_start} to {today}")
    else:
        fetch_start = HISTORICAL_START
        print("Table exists but is empty — fetching full history")
except Exception:
    fetch_start = HISTORICAL_START
    print(f"No existing table — fetching full history from {HISTORICAL_START} to {today}")

# Skip fetching if already up to date 
if fetch_start > today:
    print(f"\nAlready up to date! Latest data covers through today.")
    new_data_fetched = False
else:
    new_data_fetched = True
    print(f"\nFetching {(today - fetch_start).days + 1} day(s) of new data...")

    for borough_name, coords in boroughs.items():
        print(f"\nFetching {borough_name}: {fetch_start} to {today}...")

        params = {
            "latitude":   coords["latitude"],
            "longitude":  coords["longitude"],
            "start_date": fetch_start.isoformat(),
            "end_date":   today.isoformat(),
            "hourly": ",".join([
                "temperature_2m","apparent_temperature","relative_humidity_2m",
                "dew_point_2m","precipitation","rain","snowfall","snow_depth",
                "wind_speed_10m","wind_direction_10m","wind_gusts_10m",
                "cloud_cover","pressure_msl","weather_code"
            ]),
            "timezone": "America/New_York"
        }

        raw_weather = fetch_with_retry("https://archive-api.open-meteo.com/v1/archive", params)

        # Save raw JSON for this fetch (audit trail)
        file_path = f"{PATHS['bronze_weather']}/{borough_name.lower()}_{fetch_start.isoformat()}_to_{today.isoformat()}.json"
        mssparkutils.fs.put(file_path, json.dumps(raw_weather, indent=2), overwrite=True)
        print(f"  Saved: {file_path}")

    print(f"\nAll {len(boroughs)} boroughs fetched")

StatementMeta(, 35d9bb74-2302-4cc6-8a16-6eee6d3f81e1, 6, Finished, Available, Finished, False)

Existing bronze_weather_nyc: 13,200 rows
Latest data: 2026-04-20 23:00:00
Will fetch from: 2026-04-21 to 2026-04-21

Fetching 1 day(s) of new data...

Fetching Manhattan: 2026-04-21 to 2026-04-21...
Success on attempt 1
  Saved: Files/project_data/bronze/weather/manhattan_2026-04-21_to_2026-04-21.json

Fetching Brooklyn: 2026-04-21 to 2026-04-21...
Success on attempt 1
  Saved: Files/project_data/bronze/weather/brooklyn_2026-04-21_to_2026-04-21.json

Fetching Queens: 2026-04-21 to 2026-04-21...
Success on attempt 1
  Saved: Files/project_data/bronze/weather/queens_2026-04-21_to_2026-04-21.json

Fetching Bronx: 2026-04-21 to 2026-04-21...
Success on attempt 1
  Saved: Files/project_data/bronze/weather/bronx_2026-04-21_to_2026-04-21.json

Fetching StatenIsland: 2026-04-21 to 2026-04-21...
Success on attempt 1
  Saved: Files/project_data/bronze/weather/statenisland_2026-04-21_to_2026-04-21.json

All 5 boroughs fetched


### Phase 3: Validation

In [5]:
def ingest_weather_to_bronze(raw_json_path, borough_name, ingest_run_id):
    df_raw = spark.read.option("multiline", "true").json(raw_json_path)

    required_top_level = ["latitude", "longitude", "timezone", "hourly"]
    missing_top_level = [f for f in required_top_level if f not in df_raw.columns]

    if missing_top_level:
        print(f"    Missing top-level fields: {missing_top_level}")
        return spark.createDataFrame([], df_raw.schema), df_raw

    df = df_raw.select(
        F.explode(F.arrays_zip(
            F.col("hourly.time"),
            F.col("hourly.temperature_2m"),
            F.col("hourly.apparent_temperature"),
            F.col("hourly.relative_humidity_2m"), 
            F.col("hourly.dew_point_2m"),
            F.col("hourly.precipitation"),
            F.col("hourly.rain"),
            F.col("hourly.snowfall"),
            F.col("hourly.snow_depth"),
            F.col("hourly.wind_speed_10m"),
            F.col("hourly.wind_direction_10m"),
            F.col("hourly.wind_gusts_10m"),
            F.col("hourly.cloud_cover"),
            F.col("hourly.pressure_msl"),
            F.col("hourly.weather_code")
        )).alias("hourly_data"),
        "latitude", "longitude", "timezone", "timezone_abbreviation", "elevation"
    ).select(
        F.col("hourly_data.time").alias("timestamp"),
        F.col("hourly_data.temperature_2m").alias("temperature_2m"),
        F.col("hourly_data.apparent_temperature").alias("apparent_temperature"),
        F.col("hourly_data.relative_humidity_2m").alias("relative_humidity_2m"),
        F.col("hourly_data.dew_point_2m").alias("dew_point_2m"),
        F.col("hourly_data.precipitation").alias("precipitation"),
        F.col("hourly_data.rain").alias("rain"),
        F.col("hourly_data.snowfall").alias("snowfall"),
        F.col("hourly_data.snow_depth").alias("snow_depth"),
        F.col("hourly_data.wind_speed_10m").alias("wind_speed_10m"),
        F.col("hourly_data.wind_direction_10m").alias("wind_direction_10m"),
        F.col("hourly_data.wind_gusts_10m").alias("wind_gusts_10m"),
        F.col("hourly_data.cloud_cover").alias("cloud_cover"),
        F.col("hourly_data.pressure_msl").alias("pressure_msl"),
        F.col("hourly_data.weather_code").alias("weather_code"),
        "latitude", "longitude", "timezone", "timezone_abbreviation", "elevation"
    )

    df = df.withColumn("timestamp", F.to_timestamp("timestamp")) \
           .withColumn("temperature_2m", F.col("temperature_2m").cast("double")) \
           .withColumn("apparent_temperature", F.col("apparent_temperature").cast("double")) \
           .withColumn("relative_humidity_2m", F.col("relative_humidity_2m").cast("double")) \
           .withColumn("dew_point_2m", F.col("dew_point_2m").cast("double")) \
           .withColumn("precipitation", F.col("precipitation").cast("double")) \
           .withColumn("rain", F.col("rain").cast("double")) \
           .withColumn("snowfall", F.col("snowfall").cast("double")) \
           .withColumn("snow_depth", F.col("snow_depth").cast("double")) \
           .withColumn("wind_speed_10m", F.col("wind_speed_10m").cast("double")) \
           .withColumn("wind_direction_10m", F.col("wind_direction_10m").cast("double")) \
           .withColumn("wind_gusts_10m", F.col("wind_gusts_10m").cast("double")) \
           .withColumn("cloud_cover", F.col("cloud_cover").cast("double")) \
           .withColumn("pressure_msl", F.col("pressure_msl").cast("double")) \
           .withColumn("weather_code", F.col("weather_code").cast("int")) \
           .withColumn("latitude", F.col("latitude").cast("double")) \
           .withColumn("longitude", F.col("longitude").cast("double")) \
           .withColumn("elevation", F.col("elevation").cast("double"))

    df = df.withColumn("schema_valid", F.lit(True))
    df = df.withColumn("schema_valid",
        F.when(F.col("timezone") != "America/New_York", F.lit(False))
        .otherwise(F.col("schema_valid")))

    critical_fields = ["timestamp", "temperature_2m", "precipitation", "wind_speed_10m", "weather_code"]
    for field in critical_fields:
        df = df.withColumn("schema_valid",
            F.when(F.col(field).isNull(), F.lit(False))
            .otherwise(F.col("schema_valid")))

    df = df.withColumn("borough", F.lit(borough_name)) \
           .withColumn("source", F.lit("open-meteo-archive")) \
           .withColumn("ingest_run_id", F.lit(ingest_run_id)) \
           .withColumn("ingested_at", F.current_timestamp()) \
           .withColumn("ingest_date", F.current_date()) \

    good = df.filter(F.col("schema_valid") == True).drop("schema_valid")
    quarantine = df.filter(F.col("schema_valid") == False)

    return good, quarantine

StatementMeta(, 35d9bb74-2302-4cc6-8a16-6eee6d3f81e1, 7, Finished, Available, Finished, False)

### Phase 3b: Deduplication logic and append to Delta Table 

In [6]:
if not new_data_fetched:
    print("No new data to ingest — skipping.")
else:
    all_good = []
    all_quarantine = []

    for borough_name in ["manhattan", "brooklyn", "queens", "bronx", "statenisland"]:
        raw_file = f"{PATHS['bronze_weather']}/{borough_name}_{fetch_start.isoformat()}_to_{today.isoformat()}.json"
        print(f"\nProcessing {borough_name}...")

        good, quarantine = ingest_weather_to_bronze(raw_file, borough_name, ingest_run_id)

        good_count = good.count()
        quarantine_count = quarantine.count()

        print(f"   Good: {good_count:,} rows")
        print(f"   Quarantine: {quarantine_count:,} rows")

        all_good.append(good)
        if quarantine_count > 0:
            all_quarantine.append(quarantine)

    # Write good rows
    if all_good:
        df_new_good = all_good[0]
        for df in all_good[1:]:
            df_new_good = df_new_good.union(df)

        # Dedup within the new batch itself
        df_new_good = df_new_good.dropDuplicates(["borough", "timestamp"])
        new_good_count = df_new_good.count()

        print(f"\nAppending {new_good_count:,} new good rows to bronze_weather_nyc...")

        df_new_good.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .partitionBy("ingest_date") \
            .saveAsTable("bronze_weather_nyc")

        # Safety net: dedup the entire table after append 
        # This catches any overlap if the API returned a partial day we already had
        print("Running dedup safety net on full table...")
        df_full = spark.table("bronze_weather_nyc")
        before_dedup = df_full.count()

        w = Window.partitionBy("borough", "timestamp").orderBy(F.desc("ingested_at"))
        df_deduped = df_full.withColumn("_rn", F.row_number().over(w)) \
                            .filter(F.col("_rn") == 1) \
                            .drop("_rn")
        after_dedup = df_deduped.count()

        if before_dedup != after_dedup:
            dupes = before_dedup - after_dedup
            print(f"  Found {dupes:,} duplicates — rewriting clean table...")
            df_deduped.write \
                .format("delta") \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .partitionBy("ingest_date") \
                .saveAsTable("bronze_weather_nyc")
            print(f"  Table rewritten: {after_dedup:,} rows (removed {dupes:,} duplicates)")
        else:
            print(f"  No duplicates found — table is clean ({after_dedup:,} rows)")

        print("Bronze good table updated")

    # Write quarantine rows
    if all_quarantine:
        df_new_quarantine = all_quarantine[0]
        for df in all_quarantine[1:]:
            df_new_quarantine = df_new_quarantine.union(df)

        total_quarantine = df_new_quarantine.count()
        print(f"\nWriting {total_quarantine:,} quarantine rows...")
        df_new_quarantine.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .partitionBy("ingest_date") \
            .saveAsTable("bronze_quarantine_weather_nyc")
        print("Bronze quarantine table updated")

    print(f"\nWeather ingestion complete! Run ID: {ingest_run_id}")

StatementMeta(, 35d9bb74-2302-4cc6-8a16-6eee6d3f81e1, 8, Finished, Available, Finished, False)


Processing manhattan...
   Good: 24 rows
   Quarantine: 0 rows

Processing brooklyn...
   Good: 24 rows
   Quarantine: 0 rows

Processing queens...
   Good: 24 rows
   Quarantine: 0 rows

Processing bronx...
   Good: 24 rows
   Quarantine: 0 rows

Processing statenisland...
   Good: 24 rows
   Quarantine: 0 rows

Appending 120 new good rows to bronze_weather_nyc...
Running dedup safety net on full table...
  No duplicates found — table is clean (13,320 rows)
Bronze good table updated

Weather ingestion complete! Run ID: run_25c8f7d1-d653-4bb2-9672-53cee3d6739f


### Phase 4: QA Validation

In [7]:
print("=" * 80)
print("QA CHECK: bronze_weather_nyc")
print("=" * 80)

# 1. Total count and integrity
spark.sql("""
    SELECT
        COUNT(*) as total_records,
        COUNT(DISTINCT borough) as unique_boroughs,
        COUNT(DISTINCT ingest_run_id) as ingestion_runs
    FROM bronze_weather_nyc
""").show()

# 2. Records by borough
print("Records by borough:")
spark.sql("""
    SELECT
        borough,
        COUNT(*) as record_count,
        MIN(timestamp) as earliest_time,
        MAX(timestamp) as latest_time
    FROM bronze_weather_nyc
    GROUP BY borough
    ORDER BY record_count DESC
""").show()

# 3. Null checks
print("Null checks (should all be 0):")
spark.sql("""
    SELECT
        SUM(CASE WHEN timestamp IS NULL THEN 1 ELSE 0 END) as null_timestamp,
        SUM(CASE WHEN temperature_2m IS NULL THEN 1 ELSE 0 END) as null_temperature,
        SUM(CASE WHEN precipitation IS NULL THEN 1 ELSE 0 END) as null_precipitation,
        SUM(CASE WHEN wind_speed_10m IS NULL THEN 1 ELSE 0 END) as null_wind_speed,
        SUM(CASE WHEN borough IS NULL THEN 1 ELSE 0 END) as null_borough
    FROM bronze_weather_nyc
""").show()

# 4. Duplicate check (should return 0 rows)
print("Duplicate check (should be empty):")
spark.sql("""
    SELECT borough, timestamp, COUNT(*) as n
    FROM bronze_weather_nyc
    GROUP BY borough, timestamp
    HAVING COUNT(*) > 1
""").show()

# 5. Expected row count check
expected_hours = (today - datetime.date(2026, 1, 1)).days * 24 + 24
expected_total = expected_hours * 5
actual_total = spark.table("bronze_weather_nyc").count()
print(f"Expected ~{expected_total:,} rows (5 boroughs x ~{expected_hours:,} hours)")
print(f"Actual: {actual_total:,} rows")
if actual_total > expected_total * 1.05:
    print("WARNING: Row count is >5% above expected — check for duplicates")
elif actual_total < expected_total * 0.95:
    print("WARNING: Row count is >5% below expected — check for missing data")
else:
    print("Row count looks correct")

StatementMeta(, 35d9bb74-2302-4cc6-8a16-6eee6d3f81e1, 9, Finished, Available, Finished, False)

QA CHECK: bronze_weather_nyc
+-------------+---------------+--------------+
|total_records|unique_boroughs|ingestion_runs|
+-------------+---------------+--------------+
|        13320|              5|             2|
+-------------+---------------+--------------+

Records by borough:
+------------+------------+-------------------+-------------------+
|     borough|record_count|      earliest_time|        latest_time|
+------------+------------+-------------------+-------------------+
|       bronx|        2664|2026-01-01 00:00:00|2026-04-21 23:00:00|
|statenisland|        2664|2026-01-01 00:00:00|2026-04-21 23:00:00|
|   manhattan|        2664|2026-01-01 00:00:00|2026-04-21 23:00:00|
|      queens|        2664|2026-01-01 00:00:00|2026-04-21 23:00:00|
|    brooklyn|        2664|2026-01-01 00:00:00|2026-04-21 23:00:00|
+------------+------------+-------------------+-------------------+

Null checks (should all be 0):
+--------------+----------------+------------------+---------------+---